In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [2]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

In [3]:
train_path = Path("../data/processed/train.csv")
test_path = Path("../data/processed/test.csv")

In [4]:
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

In [5]:
X_train = train_df.drop(columns=["target_speed"])
y_train = train_df["target_speed"]

In [6]:
X_test = test_df.drop(columns=["target_speed"])
y_test = test_df["target_speed"]

In [11]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [12]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Decision Tree": DecisionTreeRegressor(max_depth=10, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42),
    
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1),
    "LightGBM": LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1),
    "CatBoost": CatBoostRegressor(iterations=100, learning_rate=0.1, random_state=42, verbose=0)
}

In [13]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    
    results.append({
        "Model": name,
        "R2 Score": round(r2, 4),
        "RMSE (m/s)": round(rmse, 4),
        "MAE (m/s)": round(mae, 4)
    })

benchmark_df = pd.DataFrame(results).sort_values(by="R2 Score", ascending=False)
benchmark_df

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001770 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1281
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 7
[LightGBM] [Info] Start training from score 5.762188


,Model,R2 Score,RMSE (m/s),MAE (m/s)
3,Random Forest,0.9565,1.1246,0.7838
5,XGBoost,0.9561,1.1293,0.7937
6,LightGBM,0.9560,1.1308,0.7895
4,Gradient Boosting,0.9532,1.1670,0.8213
7,CatBoost,0.9520,1.1815,0.8303
2,Decision Tree,0.9476,1.2339,0.8409
0,Linear Regression,0.6804,3.0485,2.3347
1,Ridge Regression,0.6804,3.0486,2.3348
